# Task 3 — Data Cleaning and Text Preprocessing

**Objective:** prepare the raw TED Talks dataset for text vectorization while preserving the original columns and not discarding talks that lack transcripts.

This notebook creates `data/processed/ted_talks_cleaned.csv`. The original CSV under `data/raw/` is never overwritten.

In [1]:
import ast
import re
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path
import pandas as pd


def find_project_root():
    """Find the project folder from the current Jupyter working directory."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "ted_talks_merged.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find data/raw/ted_talks_merged.csv. "
        "Open Jupyter from the TED_Talks_Recommendation_Ajay project folder."
    )


PROJECT_DIR = find_project_root()
RAW_DATA_PATH = PROJECT_DIR / "data" / "raw" / "ted_talks_merged.csv"
PROCESSED_DATA_PATH = PROJECT_DIR / "data" / "processed" / "ted_talks_cleaned.csv"
print("Project folder:", PROJECT_DIR)

df_raw = pd.read_csv(RAW_DATA_PATH)
print("Original shape:", df_raw.shape)
display(df_raw.head(3))

Project folder: C:\Users\Ajay\Desktop\TED_Talks_Recommendation_Ajay
Original shape: (2550, 18)


,comments,description,duration,event,film_date,languages,main_speaker,name,num_speaker,published_date,ratings,related_talks,speaker_occupation,tags,title,url,views,transcript
0,4553,Sir Ken Robinson makes an entertaining and pro...,1164,TED2006,1140825600,60,Ken Robinson,Ken Robinson: Do schools kill creativity?,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 19645}, {...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/im...",Author/educator,"['children', 'creativity', 'culture', 'dance',...",Do schools kill creativity?,https://www.ted.com/talks/ken_robinson_says_sc...,47227110,Good morning. How are you?(Laughter)It's been ...
1,265,With the same humor and humanity he exuded in ...,977,TED2006,1140825600,43,Al Gore,Al Gore: Averting the climate crisis,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 544}, {'i...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/im...",Climate advocate,"['alternative energy', 'cars', 'climate change...",Averting the climate crisis,https://www.ted.com/talks/al_gore_on_averting_...,3200520,"Thank you so much, Chris. And it's truly a gre..."
2,124,New York Times columnist David Pogue takes aim...,1286,TED2006,1140739200,26,David Pogue,David Pogue: Simplicity sells,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 964}, {'i...","[{'id': 1725, 'hero': 'https://pe.tedcdn.com/i...",Technology columnist,"['computers', 'entertainment', 'interface desi...",Simplicity sells,https://www.ted.com/talks/david_pogue_says_sim...,1636292,"(Music: ""The Sound of Silence,"" Simon & Garfun..."


## 1. Cleaning helpers

The original fields are kept. New columns ending in `_clean` are used for modelling. The tag parser accepts the list-as-text format found in this CSV.

In [3]:
def parse_tags(value):
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    if pd.isna(value):
        return []
    text = str(value).strip()
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (list, tuple, set)):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except (ValueError, SyntaxError):
        pass
    return [item.strip().strip("'\"") for item in text.split(",") if item.strip()]


def clean_text(value):
    """Lowercase, normalize whitespace, and replace most punctuation with spaces."""
    if pd.isna(value):
        return ""
    text = str(value).lower()
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"[^\w\s'-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_tags(value):
    return " ".join(clean_text(tag) for tag in parse_tags(value) if clean_text(tag))

## 2. Check identifiers and duplicates

There is no `id` or `talk_id` column in the uploaded data. The `name` slug is present and unique in this version, so it is used to create a compact `talk_id`. The URL is kept as the public link. If a future dataset does not have unique names, the code falls back to the URL.

In [4]:
if "name" in df_raw.columns and df_raw["name"].notna().all() and df_raw["name"].astype(str).is_unique:
    id_source = "name"
elif "url" in df_raw.columns and df_raw["url"].notna().all() and df_raw["url"].astype(str).is_unique:
    id_source = "url"
else:
    raise ValueError("No fully populated unique name or URL column is available for talk_id.")

print("talk_id source column:", id_source)
print("Exact duplicate rows before cleaning:", df_raw.duplicated().sum())
if "url" in df_raw.columns:
    print("Duplicate URLs before cleaning:", df_raw["url"].duplicated().sum())

talk_id source column: name
Exact duplicate rows before cleaning: 0
Duplicate URLs before cleaning: 0


## 3. Apply cleaning decisions

- Do not drop a talk just because `transcript` is missing.
- Replace missing text in modelling columns with an empty string.
- Keep optional missing metadata such as `speaker_occupation` unchanged in the original column.
- Remove repeated records only if they share the same unique URL. The supplied data has no repeated URLs, so no rows should be removed.

In [5]:
df_clean = df_raw.copy()

# Verify duplicates using URL when available. Do not deduplicate by title alone.
rows_before = len(df_clean)
if "url" in df_clean.columns:
    df_clean = df_clean.drop_duplicates(subset=["url"], keep="first").copy()
else:
    df_clean = df_clean.drop_duplicates().copy() 
rows_after = len(df_clean) 

# Create stable talk identifiers without pretending the source supplied a numeric ID.
df_clean["talk_id"] = df_clean[id_source].astype(str).str.strip()

for column in ["title", "description", "transcript", "main_speaker", "event"]:
    if column in df_clean.columns:
        df_clean[f"{column}_clean"] = df_clean[column].apply(clean_text)

if "tags" in df_clean.columns:
    df_clean["tags_clean"] = df_clean["tags"].apply(clean_tags)
else:
    df_clean["tags_clean"] = ""

# Build a single content field from whichever columns are actually available.
text_columns = [column for column in ["title_clean", "tags_clean", "description_clean", "transcript_clean"] if column in df_clean.columns]
df_clean["combined_text"] = df_clean[text_columns].fillna("").agg(" ".join, axis=1).str.replace(r"\s+", " ", regex=True).str.strip()

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed after URL duplicate check:", rows_before - rows_after)
print("Unique talk_id values:", df_clean["talk_id"].nunique())
print("Missing original transcripts retained:", int(df_clean["transcript"].isna().sum()) if "transcript" in df_clean else "Transcript column not present")
print("Blank combined_text rows:", int((df_clean["combined_text"].str.strip() == "").sum()))

Rows before: 2550
Rows after: 2550
Rows removed after URL duplicate check: 0
Unique talk_id values: 2550
Missing original transcripts retained: 86
Blank combined_text rows: 0


## 4. Review original and cleaned text side by side

In [6]:
preview_columns = [column for column in ["title", "title_clean", "tags", "tags_clean", "description", "description_clean", "transcript", "transcript_clean"] if column in df_clean.columns]
display(df_clean[preview_columns].head(3))

,title,title_clean,tags,tags_clean,description,description_clean,transcript,transcript_clean
0,Do schools kill creativity?,do schools kill creativity,"['children', 'creativity', 'culture', 'dance',...",children creativity culture dance education pa...,Sir Ken Robinson makes an entertaining and pro...,sir ken robinson makes an entertaining and pro...,Good morning. How are you?(Laughter)It's been ...,good morning how are you laughter it's been gr...
1,Averting the climate crisis,averting the climate crisis,"['alternative energy', 'cars', 'climate change...",alternative energy cars climate change culture...,With the same humor and humanity he exuded in ...,with the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",thank you so much chris and it's truly a great...
2,Simplicity sells,simplicity sells,"['computers', 'entertainment', 'interface desi...",computers entertainment interface design media...,New York Times columnist David Pogue takes aim...,new york times columnist david pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",music the sound of silence simon garfunkel hel...


## 5. Save the processed dataset

In [7]:
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(PROCESSED_DATA_PATH, index=False)
print("Saved cleaned dataset to:", PROCESSED_DATA_PATH)
print("Saved shape:", df_clean.shape)

Saved cleaned dataset to: C:\Users\Ajay\Desktop\TED_Talks_Recommendation_Ajay\data\processed\ted_talks_cleaned.csv
Saved shape: (2550, 26)


## 6. Preprocessing summary

The processed file contains the original columns plus `talk_id`, the available `*_clean` text columns, `tags_clean`, and `combined_text`. The raw file remains unchanged. Missing transcripts become empty strings only in `transcript_clean` and `combined_text`; the original `transcript` field remains missing in those 86 rows.